# Neural Style Transfer Demo

This notebook demonstrates neural style transfer using the Gatys et al. optimization-based approach.

**Features:**
- Classic Gatys style transfer
- Optimized for 8GB Mac systems
- MPS (Metal Performance Shaders) support

## Setup

First, let's import the necessary modules and check our device.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from PIL import Image
import matplotlib.pyplot as plt

from utils.image_utils import load_image, get_device
from models.gatys import GatysStyleTransfer

# Check device
device = get_device()
print(f"Using device: {device}")

if device.type == 'mps':
    print("✓ Apple Metal Performance Shaders enabled!")
elif device.type == 'cuda':
    print(f"✓ CUDA enabled! GPU: {torch.cuda.get_device_name()}")
else:
    print("⚠ Running on CPU (will be slower)")

## Load Images

Load your content and style images. For 8GB systems, we recommend using images no larger than 512x512.

In [ ]:
# Image paths - update these to your images
CONTENT_PATH = "../data/content/your_photo.jpg"
STYLE_PATH = "../data/style/starry_night.jpg"

# Maximum image size (reduce if you run out of memory)
MAX_SIZE = 512

# Load images
content_image = load_image(CONTENT_PATH, MAX_SIZE)
style_image = load_image(STYLE_PATH, MAX_SIZE)

print(f"Content image size: {content_image.size}")
print(f"Style image size: {style_image.size}")

# Display images
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

ax1.imshow(content_image)
ax1.set_title("Content Image")
ax1.axis('off')

ax2.imshow(style_image)
ax2.set_title("Style Image")
ax2.axis('off')

plt.tight_layout()
plt.show()

## Style Transfer Settings

Configure the style transfer parameters:

- **content_weight**: Higher values preserve more content structure
- **style_weight**: Higher values apply more style (try 1e5 to 1e7)
- **num_steps**: More steps = better quality, but slower

In [ ]:
# Style transfer settings
CONTENT_WEIGHT = 1
STYLE_WEIGHT = 1000000  # 1e6
NUM_STEPS = 300

# Initialize style transfer
style_transfer = GatysStyleTransfer(
    content_weight=CONTENT_WEIGHT,
    style_weight=STYLE_WEIGHT,
    num_steps=NUM_STEPS
)

## Run Style Transfer

This will take a few minutes depending on your hardware and settings.

In [ ]:
# Run style transfer
print("Starting style transfer...")
print("This may take a few minutes.\n")

result = style_transfer.transfer(
    content_image=content_image,
    style_image=style_image,
    init_image="content",  # Start from content image
    optimizer_type="lbfgs"  # L-BFGS is faster for style transfer
)

print("\n✓ Style transfer complete!")

## View Results

Compare the original content image with the stylized result.

In [ ]:
# Display results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(content_image)
axes[0].set_title("Content")
axes[0].axis('off')

axes[1].imshow(style_image)
axes[1].set_title("Style")
axes[1].axis('off')

axes[2].imshow(result)
axes[2].set_title("Result")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Save Result

In [ ]:
# Save the result
OUTPUT_PATH = "../data/output/stylized_result.jpg"
result.save(OUTPUT_PATH)
print(f"Result saved to: {OUTPUT_PATH}")

## Memory Tips

If you run out of memory on an 8GB Mac:

1. **Reduce image size**: Try `MAX_SIZE = 256` or `MAX_SIZE = 384`
2. **Fewer steps**: Use `NUM_STEPS = 100` or `NUM_STEPS = 200`
3. **Restart kernel**: Clear memory by restarting the Jupyter kernel
4. **Close other apps**: Free up RAM by closing browsers and other applications